#### 1334. Find the City With the Smallest Number of Neighbors at a Threshold Distance
* https://leetcode.com/problems/find-the-city-with-the-smallest-number-of-neighbors-at-a-threshold-distance/description/

In [1]:
# Check with Interviewer if graph is dense and nodes are less then say 100 then go with Floyd-Warshall else Dijkstra's.
# why because of time complexity - above
# In worst case
# TC of FW - V^3
# TC of Dij - E≈V^2, O(V⋅V^2 logV)=O(V^3 logV)
# | Algorithm          | Time Complexity | Better?   |
# | ------------------ | --------------- | -------   |
# | Floyd–Warshall     | (O(V^3))        | ✅        |
# | Dijkstra (n times) | (O(V^3 \log V)) | ❌        |
# SC  - FW - O(V^2) # Matrix
# SC - Dij - O(V+E), worst case for dense graph - V^2

import heapq
from typing import List
from collections import defaultdict

class Solution:
    def findTheCity(
        self,
        n: int,
        edges: List[List[int]],
        distanceThreshold: int
    ) -> int:
        """
        Optimized solution using Dijkstra from each node.

        Time Complexity: O(n * E log V)
        Space Complexity: O(E + V)
        """

        # Step 1: Build adjacency list
        graph = defaultdict(list)
        for src, dest, weight in edges:
            graph[src].append((dest, weight))
            graph[dest].append((src, weight))

        def dijkstra(start: int) -> int:
            """
            Returns number of reachable nodes within threshold
            from 'start' node.
            """
            min_distance = [float('inf')] * n
            min_distance[start] = 0

            min_heap = [(0, start)]  # (distance, node)

            while min_heap:
                current_dist, node = heapq.heappop(min_heap)

                # Skip stale entries
                if current_dist > min_distance[node]:
                    continue

                for neighbor, weight in graph[node]:
                    new_dist = current_dist + weight

                    if new_dist < min_distance[neighbor]:
                        min_distance[neighbor] = new_dist
                        heapq.heappush(min_heap, (new_dist, neighbor))

            # Count reachable cities
            return sum(
                1
                for city in range(n)
                if city != start and min_distance[city] <= distanceThreshold
            )

        # Step 2: Evaluate all cities
        min_reachable_count = n
        result_city = -1

        for city in range(n):
            reachable_count = dijkstra(city)

            if reachable_count <= min_reachable_count:
                min_reachable_count = reachable_count
                result_city = city

        return result_city

In [ ]:
# (Floyd–Warshall)

from typing import List

class Solution:
    def findTheCity(
        self,
        n: int,
        edges: List[List[int]],
        distanceThreshold: int
    ) -> int:
        """
        Finds the city with the smallest number of reachable cities
        within the given distance threshold.

        If multiple cities have the same count, returns the city
        with the greatest index.

        Approach:
        - Use Floyd-Warshall to compute all-pairs shortest paths
        - Count reachable cities for each node
        - Apply tie-breaking logic

        Time Complexity: O(n^3)
        Space Complexity: O(n^2)
        """

        # Step 1: Initialize distance matrix
        INF = float('inf')
        distance = [[INF] * n for _ in range(n)]

        # Distance to self is always 0
        for city in range(n):
            distance[city][city] = 0

        # Step 2: Populate direct edges
        for src, dest, weight in edges:
            distance[src][dest] = weight
            distance[dest][src] = weight  # Undirected graph

        # Step 3: Floyd-Warshall (All-Pairs Shortest Path)
        for via in range(n):
            for src in range(n):
                for dest in range(n):
                    # Relaxation step
                    if distance[src][dest] > distance[src][via] + distance[via][dest]:
                        distance[src][dest] = distance[src][via] + distance[via][dest]

        # Step 4: Find city with minimum reachable neighbors
        min_reachable_count = n
        result_city = -1

        for city in range(n):
            reachable_count = 0

            for neighbor in range(n):
                if city != neighbor and distance[city][neighbor] <= distanceThreshold:
                    reachable_count += 1

            # Tie-breaking: prefer larger index
            if reachable_count <= min_reachable_count:
                min_reachable_count = reachable_count
                result_city = city

        return result_city